# Regressão com MLPs

As MLPs também podem ser usadas para tarefas de regressão. Se quiser predizer um valor único (por exemplo, o preço de um imóvel, levando em conta muitas de suas características), precisará somente de um único neurônio de saída: sua saída é o valor predito. Para regressão multivariada (ou seja, para predizer diversos valores de uma vez), você precisa de um neurônio de saída por dimensão de saída. 

Em geral, ao criar uma MLP para regressão, você não quer usar nenhuma função de ativação para os neurônios de saída; portanto, eles ficam livres para gerar a saída de qualquer intervalo de valores. Caso deseje garantir que a saída seja sempre positiva, poderá usar a função de ativação ReLU na camada de saída. Como alternativa, você pode utilizar a função de ativação *softplus*, uma variante suave da função ReLU:

> softplus(z) = log(1 + exp(z))

Ela fica próxima de 0 quando z é negativo e fica próxima de z quando z é positivo. Por fim, se você quiser garantir que as predições fiquem dentro de um determinado intervalo de valores, poderá empregar a função logística ou a tangente hiperbólica e escalonar os rótulos para o intervalo apropriado: 0 a 1 para a função logística e de -1 a 1 para a tangente hiperbólica.

A função de perda a ser usada durante o treinamento é normalmente o erro médio quadrático, mas, se tiver muitos outliers no conjunto de treinamento, poderá preferir utilizar o erro absoluto médio. Como alternativa, você pode usar a função perda de huber, uma combinação de ambas. 

> A perda de Huber é quadrática quando o erro é menor que um limiar L (normalmente 1), e linear quando o erro é maior que L. A parte linear a deixa menos sensível aos outliers do que o erro médio quadrático e a parte quadrática possibilita convergir mais rapidamente e ter mais precisão que o erro médio absoluto.

<p align=center>
<img src="https://github.com/vitorbeltrao/Pictures/blob/main/arq_tipica_mlp_regressao.png?raw=true" width="50%"></p>

In [1]:
# importar os pacotes necessários
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from scikeras.wrappers import KerasRegressor
from datasets import load_dataset
from scipy.stats import reciprocal


import warnings
warnings.filterwarnings('ignore')

/Users/vitorabdo/Desktop/personal_work/dl_book/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# carregar o conjunto de dados
HUGGINGFACE_API_KEY = os.environ.get("HUGGINGFACE_API_KEY")
dataset = load_dataset(
    "penikmatrumput/nasa-cmapss-rul",
    token=HUGGINGFACE_API_KEY
)

In [3]:
# ler os dados de treino e teste como um dataframe pandas
df_train = pd.DataFrame(dataset["train"])
df_test = pd.DataFrame(dataset["test"])

# verificar o tamanho dos dados de treino e teste
print("Tamanho do conjunto de dados de treino:", len(df_train))
print("Tamanho do conjunto de dados de teste:", len(df_test))

Tamanho do conjunto de dados de treino: 160359
Tamanho do conjunto de dados de teste: 104897


In [4]:
# visualizar as primeiras linhas do conjunto de dados de treino
df_train.head()

,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


## Pré-processamento dos dados

In [5]:
# criar um pipeline de pré-processamento para os dados
pipeline = Pipeline([
    ("scaling", StandardScaler())
])

In [6]:
# transformando os dados de treino, validação e teste
X_train = pipeline.fit_transform(df_train.drop(columns=["RUL"]))
y_train = df_train["RUL"].values

X_test = pipeline.transform(df_test.drop(columns=["RUL"]))
y_test = df_test["RUL"].values

In [8]:
# visualizar uma amostra dos dados transformados
feature_names = pipeline.named_steps["scaling"].get_feature_names_out()

df = pd.DataFrame(
    X_train,
    columns=feature_names
)

df.head()

,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,-1.434856,-1.464382,-1.041429,-1.115418,0.345955,1.079185,1.046626,1.037990,1.024534,1.107714,...,1.113752,0.345200,0.616065,-0.845217,0.963589,1.009022,0.801655,0.345955,1.121962,1.119494
1,-1.434856,-1.452411,-1.041272,-1.115146,0.345955,1.079185,1.054395,1.055929,1.043169,1.107714,...,1.117528,0.345649,0.527629,-0.828851,0.963589,1.009022,0.801655,0.345955,1.116830,1.120150
2,-1.434856,-1.440440,-1.041647,-1.113516,0.345955,1.079185,1.059103,1.023520,1.050946,1.107714,...,1.118380,0.345289,0.549211,-0.847479,0.963589,0.944550,0.801655,0.345955,1.112553,1.108831
3,-1.434856,-1.428470,-1.041344,-1.114331,0.345955,1.079185,1.059103,0.979517,1.033851,1.107714,...,1.121060,0.345739,0.556653,-0.913473,0.963589,1.009022,0.801655,0.345955,1.106566,1.113065
4,-1.434856,-1.416499,-1.041502,-1.114874,0.345955,1.079185,1.059574,0.980025,1.065766,1.107714,...,1.116979,0.345379,0.556281,-0.832044,0.963589,1.041258,0.801655,0.345955,1.108277,1.117413


## Modelagem

Usar a sequential API para criar, treinar, avaliar e usar uma MLP de regressão com o intuito de fazer predições é muito semelhante ao que fizemos para a classificação.

As principais diferenças são o fato de a camada de saída ter um único neurônio (visto que queremos predizer somente um único valor) e não usar nenhuma função de ativação; a função de perda é o erro qudrático médio (MSE).

In [9]:
model = keras.models.Sequential([
    # define o formato de entrada que a rede espera receber
    keras.layers.Input(shape=(X_train.shape[1],)),
    
    # camadas densas (totalmente conectadas)
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),

    # camada de saída sem função de ativação
    keras.layers.Dense(1)
])

In [10]:
# exibir todas as camadas do modelo
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,777 (46.00 KB)

 Trainable params: 11,777 (46.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# compilação do modelo
model.compile(
    optimizer="sgd",
    loss="huber",
    metrics=["mean_absolute_error"]
)

In [12]:
# treinando e avaliando o modelo
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    shuffle=True
)

Epoch 1/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 222us/step - loss: 47.3964 - mean_absolute_error: 47.8934 - val_loss: 42.2531 - val_mean_absolute_error: 42.7503
Epoch 2/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 211us/step - loss: 35.1477 - mean_absolute_error: 35.6435 - val_loss: 41.5479 - val_mean_absolute_error: 42.0451
Epoch 3/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 206us/step - loss: 33.5326 - mean_absolute_error: 34.0278 - val_loss: 41.2224 - val_mean_absolute_error: 41.7185
Epoch 4/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 205us/step - loss: 32.9612 - mean_absolute_error: 33.4563 - val_loss: 39.4294 - val_mean_absolute_error: 39.9257
Epoch 5/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 208us/step - loss: 32.5034 - mean_absolute_error: 32.9985 - val_loss: 41.0225 - val_mean_absolute_error: 41.5185
Epoch 6/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 203us/step - loss: 32.1846 - mean_absolute_error: 32.6793 - val_loss: 39.5117 - val_mean_absolute_error: 40.0074
Epoch 7/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 200us/s

O treinamento desse modelo pode durar por horas. Isso é bastante comum, principalmente quando treinamos grandes conjuntos de dados. Nesse caso, você não deve salvar seu modelo apenas no fim do treinamento, como também salvar pontos de verificação em intervalos regulares durante o treinamento, para evitar que tudo se perca, caso a sua máquina trave.

**Usando as funções de callbacks**

O método *fit()* aceita um argumento *callbacks* que permite especificar uma lista de objetos que a Keras chamará no início e no fim do treinamento, no início e no fim de cada época, e antes e depois do processamento de cada batch. Por exemplo, o callback *ModelCheckpoint* salva os pontos de verificação do modelo em intervalos regulares durante o treinamento, por padrão no fim de cada época.

In [13]:
# usando as funções de callbacks: modelcheckpoint
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    "saved_models/best_regression_model.keras", save_best_only=True
)
# history = model.fit(
#     X_train, y_train,
#     validation_split=0.2,
#     epochs=30,
#     batch_size=32,
#     shuffle=True,
#     callbacks=[checkpoint_cb]
# )

Nesse caso acima, como estamos utilizando um conjunto de validação durante o treinamento, podemos definir *save_best_only=True* ao criar o check point. Dessa forma, ele apenas salvará seu modelo quando o desempenho no conjunto de validação até então for o melhor. Desse modo, você não precisa se preocupar muito com o treinamento e com um possível overfitting: basta restaurar o último modelo salvo após o treinamento e este será o melhor modelo no conjunto de validação.

Outra callback que podemos utilizar é a *EarlyStopping*. Ela interromperá o treinamento quando não calcular nenhum progresso na validação definida para várias épocas (definida pelo argumento *patience*) e, opcionalmente, reverterá para o melhor modelo. Você pode combinar as duas callbacks para salvar os pontos de verificação do modelo e interromper o treinamento com antecedência, quando não houver mais progresso (a fim de evitar perda de tempo e recursos).

In [14]:
# usando as funções de callbacks: modelcheckpoint e earlystopping
early_stopping_cb = keras.callbacks.EarlyStopping(
    patience=10, 
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    shuffle=True,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

Epoch 1/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 208us/step - loss: 29.8482 - mean_absolute_error: 30.3423 - val_loss: 38.4106 - val_mean_absolute_error: 38.9064
Epoch 2/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 204us/step - loss: 29.8045 - mean_absolute_error: 30.2985 - val_loss: 38.8790 - val_mean_absolute_error: 39.3746
Epoch 3/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 208us/step - loss: 29.8522 - mean_absolute_error: 30.3457 - val_loss: 38.6530 - val_mean_absolute_error: 39.1494
Epoch 4/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 207us/step - loss: 29.7619 - mean_absolute_error: 30.2558 - val_loss: 40.2757 - val_mean_absolute_error: 40.7716
Epoch 5/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 202us/step - loss: 29.8007 - mean_absolute_error: 30.2946 - val_loss: 39.3862 - val_mean_absolute_error: 39.8816
Epoch 6/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 202us/step - loss: 29.6534 - mean_absolute_error: 30.1471 - val_loss: 40.2001 - val_mean_absolute_error: 40.6957
Epoch 7/100
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 

O número de épocas até pode ser definido como um valor alto, uma vez que o treinamento será interrompido automaticamente quando não progredir mais. Nesse caso, não há necessidade de restaurar o melhor modelo salvo, pois a callback da *EarlyStopping* vai rastrear os melhores pesos e os restaurar no fim do treinamento.

Se precisar de controle extra, você pode escrever as suas próprias callbacks.

**Aperfeiçoando os hiperparâmetros das redes neurais**

A flexibilidade das redes neurais também é um de seus principais empecilhos: existem muitos hiperparâmetros para ajustar. Você pode alterar o número de camadas, o número de neurônios por camada, o tipo de função de ativação a ser usada em cada camada, a lógica de inicialização de peso e muito mais. Mas como saber qual a combinação de hiperparâmetros é a melhor para sua tarefa?

Uma opção é simplesmente testar uma diversidade de combinações de hiperparâmetros e ver qual funciona melhor no conjunto de validação (ou usar a validação cruzada). Por exemplo, podemos utilizar o *GridSearchCV* ou o *RandomizedSearchCV* para explorar o espaço dos hiperparâmetros. Para tal, é necessário empacotarmos nossos modelos Keras em objetos que imitem os regressores regulares da Scikit-Learn. O primeiro passo é criar uma função que construirá e compilará um modelo Keras, levando em conta um conjunto de hiperparâmetros.

In [15]:
# função que compila modelo keras levando em conta um conjunto de hiperparametros
def build_model(n_hidden=1, n_neurons=30, learning_rate=3e-3, input_shape=10):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=(input_shape,)))

    # camadas densas (totalmente conectadas)
    for layer in range(n_hidden):
        model.add(
            keras.layers.Dense(
                n_neurons,
                activation="relu"
            )
        )

    # camada de saída sem função de ativação
    model.add(keras.layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=learning_rate
        ),
        loss="huber",
        metrics=["mean_absolute_error"]
    )

    return model

Essa função cria um modelo *Sequential* simples para regressão univariada (e somente um neurônio de saída), com o formato de entrada fornecido e o número especificado de camadas e neurônios ocultos, e os compila usando um otimizador *SGD* definido com a taxa de aprendizado especificada. É uma boa prática fornecer padrões razoáveis para tantos hiperparâmetros quanto possível, como a Scikit-Learn faz.

Agora, criaremos um *KerasRegressor* com base na função *build_model()*.

In [16]:
# criar um wrapper em torno do modelo keras
keras_reg = KerasRegressor(
    model=build_model,
    model__n_hidden=1,
    model__n_neurons=30,
    model__learning_rate=3e-3,
    model__input_shape=X_train.shape[1]
)

O objeto *KerasRegressor* é um pequeno wrapper em torno do modelo Keras criado e que usa o *build_model()*. Como não especificamos nenhum hiperparâmetro ao criá-lo, ele usará os hiperparâmetros padrão que definimos no *build_model()*. Agora, podemos usar esse objeto como um regressor da Scikit-Learn comum: podemos treiná-lo usando seu método *fit()*, depois avaliá-lo utilizando o método *score()* e usá-lo para fazer previsões por meio do método *predict()*.

In [17]:
# treinar e avaliar o modelo
history = keras_reg.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    shuffle=True,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

mse_test = keras_reg.score(X_test, y_test)
print("MSE no conjunto de teste:", mse_test)

y_pred = keras_reg.predict(X_test)
plt.scatter(y_test, y_pred)
plt.xlabel("Valores Reais")
plt.ylabel("Valores Preditos")
plt.title("Valores Reais vs. Valores Preditos")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

Epoch 1/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 184us/step - loss: 72.9976 - mean_absolute_error: 73.4957 - val_loss: 57.0694 - val_mean_absolute_error: 57.5670
Epoch 2/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 168us/step - loss: 42.8358 - mean_absolute_error: 43.3328 - val_loss: 53.4326 - val_mean_absolute_error: 53.9302
Epoch 3/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 173us/step - loss: 40.4312 - mean_absolute_error: 40.9281 - val_loss: 49.9431 - val_mean_absolute_error: 50.4402
Epoch 4/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 170us/step - loss: 38.0288 - mean_absolute_error: 38.5249 - val_loss: 45.7943 - val_mean_absolute_error: 46.2917
Epoch 5/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 172us/step - loss: 36.1330 - mean_absolute_error: 36.6287 - val_loss: 43.4200 - val_mean_absolute_error: 43.9172
Epoch 6/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 167us/step - loss: 34.6300 - mean_absolute_error: 35.1256 - val_loss: 42.9016 - val_mean_absolute_error: 43.3988
Epoch 7/30
4009/4009 ━━━━━━━━━━━━━━━━━━━━ 1s 167us/s

ValueError: Input contains NaN.

Qualquer parâmetro extra que passar para o método *fit()* será passado para o modelo Keras subjacente. Repare também que o score será o oposto do MSE, pois a Scikit-Learn quer scores e não perdas (ou seja, os maiores devem ser melhores). Não queremos treinar um único modelo como este, queremos treinar centenas de variantes e ver qual delas apresenta o melhor desempenho no conjunto de validação. Já que existem muitos hiperparâmetros, é preferível usar um randomized search em vez do grid search. 

In [ ]:
# treinar e avaliar o modelo usando o randomized search
param_distribs = {
    "model__n_hidden": [0, 1, 2, 3],
    "model__n_neurons": np.arange(1, 100),
    "model__learning_rate": reciprocal(3e-4, 3e-2)
}

rnd_search_cv = RandomizedSearchCV(
    keras_reg,
    param_distribs,
    n_iter=5,
    cv=3
)

rnd_search_cv.fit(
    X_train, y_train, 
    epochs=100, 
    validation_split=0.2,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

Epoch 1/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 1s 178us/step - loss: 126.1348 - mean_absolute_error: 126.6336 - val_loss: 129.1414 - val_mean_absolute_error: 129.6401
Epoch 2/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 0s 163us/step - loss: 122.1586 - mean_absolute_error: 122.6572 - val_loss: 125.4556 - val_mean_absolute_error: 125.9543
Epoch 3/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 0s 163us/step - loss: 118.1444 - mean_absolute_error: 118.6430 - val_loss: 121.9748 - val_mean_absolute_error: 122.4733
Epoch 4/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 0s 162us/step - loss: 114.2093 - mean_absolute_error: 114.7076 - val_loss: 118.6563 - val_mean_absolute_error: 119.1549
Epoch 5/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 0s 163us/step - loss: 111.3096 - mean_absolute_error: 111.8083 - val_loss: 115.4940 - val_mean_absolute_error: 115.9927
Epoch 6/100
2673/2673 ━━━━━━━━━━━━━━━━━━━━ 0s 161us/step - loss: 107.6315 - mean_absolute_error: 108.1300 - val_loss: 112.4595 - val_mean_absolute_error: 112.9581
Epoch 7/100
2673/2673 

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KerasRegresso...put_shape=26 )
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': <scipy.stats....t 0x32e182090>, 'model__n_hidden': [0, 1, ...], 'model__n_neurons': array([ 1, 2..., 97, 98, 99])}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examp

O *RandomizedSearchCV* usa a validação cruzada com K-fold, e não os dados reservados para validação no parâmetro *validation_split*, que são utilizadas apenas para as funções de callbacks.

Essa exploração pode durar muito tempo. Uma vez que termine, você poderá acessar os melhores parâmetros encontratos, o melhor score e o modelo Keras treinado.

In [22]:
# encontrar os melhores hiperparâmetros
print("Melhores hiperparâmetros:", rnd_search_cv.best_params_)

# encontrar o melhor score do modelo
print("Melhor score do modelo:", rnd_search_cv.best_score_)

# obter o melhor modelo treinado
best_model = rnd_search_cv.best_estimator_.model

Melhores hiperparâmetros: {'model__learning_rate': 0.005558866265342473, 'model__n_hidden': 1, 'model__n_neurons': 38}
Melhor score do modelo: 0.6000075538953146


Agora, você pode salvar esse modelo, avaliá-lo no conjunto de testes e, se ficar satisfeito com seu desempenho, implementá-lo em produção. Usar o randomized search não é muito difícil e funciona bem para uma série de problemas. Contudo, quando o treinamento é demorado (por exemplo, para problemas mais complexos com conjuntos de dados maiores), essa abordagem explora somente uma pequena parte do espaço do hiperparâmetro. Você pode mitigar esse problema parcialmente, auxiliando no processo de pesquisa de forma manual: primeiro, execute um random search rápido usando intervalos amplos de valores de hiperparâmetros, depois rode outro random search usando intervalos menores de valores centrados nos melhores intervalos encontrados durante a primeira execução e assim por diante. Espera-se que essa abordagem foque um bom conjunto de hiperparâmetros. 

Existem outras técnicas para explorar um espaço de pesquisa com mais eficiência do que de modo aleatório. A ideia é simples: quando se comprova que uma região do espaço é boa, ela deve ser mais explorada. Essas técnicas ampliam as possibilidades e levam a soluções muito melhores em menos tempo. Existem várias bibliotecas Python para que isso que você pode utilizar.

## Conclusão

A importância desse algoritmo é tão grande que vale a pena resumir novamente: para cada instância de treinamento, o algoritmo primeiro faz uma predição (forward pass) e calcula o erro, depois passa por cada camada no sentido inverso a fim de calcular a contribuição do erro de cada conexão e, por fim, ajusta os pesos da conexão para reduzir o erro (etapa do gradiente descendente).